# UAE Desert-Edge Urban Expansion & Resilience Monitor

A Colab-first EO/GIS workflow for a real planning question: where is urban expansion associated with vegetation/water trade-offs, and which locations deserve human planning review? It searches imagery through STAC, applies strict quality gates, calculates NDVI/NDWI/NDBI/EVI, plots a time series, and exports a georeferenced review surface.

**Important:** this is a research prototype. It does not measure temperature, groundwater depletion, construction, or desertification directly. Review the evidence and limitations before using any result.

In [ ]:
%pip -q install uv
!uv pip install --system -q pystac-client planetary-computer odc-stac xarray dask[array] rasterio rioxarray geopandas shapely pyproj numpy pandas matplotlib plotly folium scikit-learn pyarrow tqdm

The install cell is intentionally explicit so the notebook is self-contained when uploaded directly to Google Colab.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import folium
import rioxarray
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'uae_monitor.py').exists():
    raise FileNotFoundError('Open/upload the complete project folder so src/uae_monitor.py is available.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.uae_monitor import (
    AOIS, SearchConfig, aoi_geometry, search_sentinel_items, load_cube,
    add_indices, quality_report, monthly_composite, summarize_time_series,
    change_surface, explainable_change_score, evidence_summary,
)
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ---- Controls: strict quality defaults ----
AOI_NAME = 'Dubai urban cluster'
START = '2024-01-01'
END = '2025-12-31'
MAX_CLOUD = 10.0
MIN_VALID_PIXEL_FRACTION = 0.90
INDEX_TO_MAP = 'NDVI'
DROP_THRESHOLD = 0.15

print(AOIS[AOI_NAME])

In [ ]:
config = SearchConfig(start=START, end=END, cloud_cover_max=MAX_CLOUD)
aoi = aoi_geometry(AOI_NAME)
items = search_sentinel_items(config, aoi)
print(f'Matched {len(items)} acquisitions')
if not items:
    raise RuntimeError('No imagery matched. Widen the date range or cloud threshold.')

provenance = pd.DataFrame([{
    'id': item.id,
    'date': item.datetime,
    'cloud_cover': item.properties.get('eo:cloud_cover'),
} for item in items])
display(provenance.head(20))

In [ ]:
cube = add_indices(load_cube(items, config, aoi))
qa = quality_report(cube)
qa.to_csv(OUTPUT_DIR / 'quality_report_before_filter.csv', index=False)
display(qa.head(20))

keep_dates = qa.loc[qa['valid_pixel_fraction'] >= MIN_VALID_PIXEL_FRACTION, 'date'].values
cube = cube.sel(time=cube.time.isin(keep_dates))
if cube.sizes.get('time', 0) == 0:
    raise RuntimeError('No scenes passed the 90% valid-pixel quality gate.')

cube = monthly_composite(cube)
ts = summarize_time_series(cube)
ts.to_csv(OUTPUT_DIR / 'monthly_index_timeseries.csv', index=False)
print('Cube dimensions:', dict(cube.sizes))
display(ts.head())

In [ ]:
fig = px.line(ts, x='date', y=['NDVI','NDWI','NDBI','EVI'], markers=True,
             title=f'{AOI_NAME}: cloud-masked spectral-index time series')
fig.update_yaxes(range=[-1, 1])
fig.show()

In [ ]:
change = explainable_change_score(change_surface(cube, INDEX_TO_MAP), DROP_THRESHOLD)
print('Latest acquisition:', str(cube.time.values[-1]))
print('Flagged pixel fraction:', float(change.flagged.mean().compute()))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
change.baseline.plot(ax=axes[0], cmap='viridis', vmin=-1, vmax=1); axes[0].set_title('Baseline')
change.latest.plot(ax=axes[1], cmap='viridis', vmin=-1, vmax=1); axes[1].set_title('Latest')
change.score.plot(ax=axes[2], cmap='magma', vmin=0, vmax=100); axes[2].set_title('Explainable score')
plt.tight_layout()

In [ ]:
# Lightweight interactive map centered on the selected preset.
bbox = AOIS[AOI_NAME]['bbox']
center = [(bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2]
m = folium.Map(location=center, zoom_start=10, tiles='CartoDB positron')
folium.GeoJson(aoi, name=AOI_NAME, style_function=lambda _: {'color':'#00a6a6','fill':False,'weight':2}).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
# Optional exploratory ML: rank unusual acquisitions from the AOI-level time series.
from sklearn.ensemble import IsolationForest
features = ts[['NDVI','NDWI','NDBI','EVI']].dropna()
if len(features) >= 6:
    model = IsolationForest(random_state=42, contamination='auto')
    ts.loc[features.index, 'ml_anomaly_score'] = -model.fit_predict(features)
    display(ts.sort_values('ml_anomaly_score', ascending=False).head())
else:
    print('Fewer than six valid acquisitions: skip unsupervised ranking and rely on the transparent change score.')

In [ ]:
# Save a georeferenced raster and an evidence manifest for QGIS/GeoLibre/review.
score = change['score'].compute().rio.write_crs(config.output_crs)
score.rio.to_raster(OUTPUT_DIR / 'ndvi_review_score.tif', compress='deflate')
evidence = evidence_summary(items, config, AOI_NAME, ts)
evidence.update({'quality_gate': {'scene_cloud_cover_max': MAX_CLOUD, 'min_valid_pixel_fraction': MIN_VALID_PIXEL_FRACTION}, 'outputs': ['quality_report_before_filter.csv', 'monthly_index_timeseries.csv', 'ndvi_review_score.tif']})
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(evidence, indent=2, default=str), encoding='utf-8')
print(json.dumps(evidence, indent=2, default=str)[:8000])

## Recruiter-facing extension ideas

- Add Sentinel-1 VV/VH for cloud-independent structural change.
- Add ESA WorldCover as a land-cover context layer.
- Add a reviewed UAE validation set and report precision/recall for flags.
- Move the controls into Streamlit or Panel after the notebook is stable.
- Benchmark TorchGeo/TerraTorch only when a labelled task exists.
- Connect the evidence JSON to an LLM only behind schema validation, citations, abstention, and human review.